In [1]:
# ------------------------------------------------------------------
# SBERT KURULUM
# ------------------------------------------------------------------

!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 6.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 11.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 13.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 11.9 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: regex
    Found existing installation: regex 2024.11.6
    Uninstalling regex-2024.11.6:
      Successfully uninstalled regex-2024.11.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [sentence-transformers]ence-transformers]


In [2]:
# ------------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------------

from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# ------------------------------------------------------------------
# DATA LOAD
# ------------------------------------------------------------------

products_clean = pd.read_csv("../data_interim/products_clean.csv")

products_clean.head()

,product_id,product_name,brand_name,loves_count,rating,reviews,ingredients,price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,product_text
0,P473671,Fragrance Discovery Set,19-69,6320,3.6364,11.0,Capri Eau de Parfum: Alcohol Denat. (SD Alcoho...,35.0,0,0,1,0,0,Unisex/ Genderless Scent Warm &Spicy Scent Woo...,Fragrance,Value & Gift Sets,Perfume Gift Sets,0,Fragrance Discovery Set 19-69 Fragrance Value ...
1,P473668,La Habana Eau de Parfum,19-69,3827,4.1538,13.0,"Alcohol Denat. (SD Alcohol 39C), Parfum (Fragr...",195.0,0,0,1,0,0,Unisex/ Genderless Scent Layerable Scent Warm ...,Fragrance,Women,Perfume,2,La Habana Eau de Parfum 19-69 Fragrance Women ...
2,P473662,Rainbow Bar Eau de Parfum,19-69,3253,4.2500,16.0,"Alcohol Denat. (SD Alcohol 39C), Parfum (Fragr...",195.0,0,0,1,0,0,Unisex/ Genderless Scent Layerable Scent Woody...,Fragrance,Women,Perfume,2,Rainbow Bar Eau de Parfum 19-69 Fragrance Wome...
3,P473660,Kasbah Eau de Parfum,19-69,3018,4.4762,21.0,"Alcohol Denat. (SD Alcohol 39C), Parfum (Fragr...",195.0,0,0,1,0,0,Unisex/ Genderless Scent Layerable Scent Warm ...,Fragrance,Women,Perfume,2,Kasbah Eau de Parfum 19-69 Fragrance Women Per...
4,P473658,Purple Haze Eau de Parfum,19-69,2691,3.2308,13.0,"Alcohol Denat. (SD Alcohol 39C), Parfum (Fragr...",195.0,0,0,1,0,0,Unisex/ Genderless Scent Layerable Scent Woody...,Fragrance,Women,Perfume,2,Purple Haze Eau de Parfum 19-69 Fragrance Wome...


In [4]:
print("product_text" in products_clean.columns)

True


In [5]:
# ------------------------------------------------------------------
# SBERT MODEL
# ------------------------------------------------------------------

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# ------------------------------------------------------------------
# EMBEDDING OLUŞTURMA
# ------------------------------------------------------------------

product_texts = products_clean["product_text"].fillna("").tolist()

embeddings = model.encode(product_texts, show_progress_bar=True)

Batches:   0%|          | 0/266 [00:00<?, ?it/s]

In [7]:
# ------------------------------------------------------------------
# SIMILARITY MATRIX
# ------------------------------------------------------------------

similarity_matrix_sbert = cosine_similarity(embeddings)

In [8]:
# ------------------------------------------------------------------
# INDEX MAP
# ------------------------------------------------------------------

productid_to_index = pd.Series(products_clean.index, index=products_clean["product_id"]).to_dict()

index_to_productid = dict(zip(products_clean.index, products_clean["product_id"]))

In [9]:
# ------------------------------------------------------------------
# SBERT CONTENT RECOMMENDER
# ------------------------------------------------------------------

def content_model_sbert(product_id, top_n=10):

    if product_id not in productid_to_index:
        print("Ürün bulunamadı")
        return None

    idx = productid_to_index[product_id]

    similarity_scores = list(enumerate(similarity_matrix_sbert[idx]))

    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similarity_scores = similarity_scores[1:]  # kendisini çıkar

    top_items = similarity_scores[:top_n]

    results = []

    for i, score in top_items:
        results.append({
            "product_id": index_to_productid[i],
            "similarity_score": score
        })

    return pd.DataFrame(results)

In [10]:
content_model_sbert("P422077", top_n=10)

,product_id,similarity_score
0,P377873,0.960420
1,P505043,0.901596
2,P476011,0.855837
3,P482262,0.849628
4,P503996,0.844385
5,P456151,0.835054
6,P475045,0.834300
7,P504805,0.832148
8,P416200,0.829899
9,P505129,0.826264
